# Capítulo 9: Event-Driven — Earnings Surprise (PEAD)

**Estrategia:** Comprar acciones cuya sorpresa de beneficios (EPS real vs. consenso) supera un umbral, **confirmada** por la reacción del precio (exceso sobre SPY) y por volumen anómalo en el anuncio. Se mantiene durante la ventana de *drift* o hasta justo antes del siguiente anuncio, con stop-loss y un máximo de posiciones.

**Advertencias:** (1) El universo son grandes compañías *actuales* de EE. UU. → sesgo de supervivencia. (2) `yfinance` no es una base de datos de consenso profesional: las fechas y sorpresas pueden tener huecos. (3) La hora exacta del anuncio no es fiable, por eso la entrada se retrasa a T+1.

**Objetivo:** Medir si el *post-earnings announcement drift* genera alpha frente al Buy & Hold del universo y frente a SPY.

### Registro de cambios respecto a la versión anterior

| # | Problema | Corrección |
|---|----------|-----------|
| 1 | `FECHA_INICIO/FECHA_FIN = None` → `TypeError` en Bloque 8 y en el análisis (`None[:4]`, y `str - str`) | Se rellenan con las fechas del menú (Bloque 3) y los títulos/CAGR usan el índice real de la serie |
| 2 | Cabecera del notebook era del Cap. 8 (Merger Arbitrage) | Corregida |
| 3 | `STOP_LOSS_PCT` y `estado['posiciones']` se definían pero **nunca se usaban** | Stop-loss real desde el precio de entrada, con bloqueo hasta el siguiente evento |
| 4 | Filtro de volumen evaluado **cada día** de la ventana (y sobrescrito entre eventos) → posición intermitente | Un único test por evento (pico de volumen en T/T+1 vs. media de las 20 sesiones previas) |
| 5 | Señal activa desde el día del anuncio (look-ahead con anuncios *after-close*) | Señal desde T+1, tras conocer la reacción completa |
| 6 | `earnings_dates` devuelve solo ~12 trimestres | `get_earnings_dates(limit=...)` |
| 7 | `mask.sum()` mostraba solo el último evento (y `NameError` sin eventos) | Resumen por activo mediante embudo de filtros |
| 8 | "Días con posición" = `(patrimonio > 0).sum()` (siempre 100%) | Exposición calculada desde las señales |
| 9 | Peso 1/n entre las señales activas (con 1 señal → 100% en una acción) | Máx. `MAX_POSICIONES`, peso fijo 1/MAX, ranking por sorpresa |
| 10 | Se mantenía la posición durante el siguiente anuncio (riesgo binario) | Salida `MARGEN_PRE_EARNINGS` sesiones antes |
| 11 | Ventana de 60 días *naturales* (≈42 sesiones) | 60 **sesiones** bursátiles |
| 12 | Gráfico con doble eje Y (curvas no comparables visualmente) | Un único eje, todo normalizado al capital inicial |
| 13 | Texto de conclusiones decía "umbral >5%" con umbral real 2% | Texto generado con los parámetros reales |

### Versión 6: cinco líneas de investigación

| Sugerencia | Implementación |
|---|---|
| Ampliar a mid-caps | `UNIVERSO_MID` (24 compañías) además de `UNIVERSO_LARGE`; `UNIVERSO_ACTIVO` = `'large'`, `'mid'` o `'ambos'`; el Bloque 5 muestra la capitalización actual para verificar la clasificación |
| Costes realistas | Celda de costes: barrido de niveles, rotación anual, coste anual, coste "mezcla" según el reparto grandes/mid y coste de equilibrio frente a SPY |
| Filtro de momentum pre-anuncio | MA50 > MA200 evaluado en el cierre de T-1 (`calcular_tendencia`, equivalente vectorizado de `filtro_momentum`) |
| Tamaño dinámico por sorpresa | `TRAMOS_PESO` (15 % / 10 % / 5 %), máx. 8 posiciones, reescalado si la suma supera el 100 % |
| Análisis de subgrupos | Sector, tamaño, régimen alcista/bajista, trimestre del anuncio, tramo de sorpresa y tendencia previa; más captura de la cartera por régimen |
| Comparativa | Celda de variantes (12 combinaciones) para ver qué aporta cada cambio |

Descubrimiento sobre el motor: `coste_transaccion` se aplica a la **rotación unilateral** (Σ|Δpeso|/2), así que 0,10 % equivale a unos 5 pb por lado.

### Versión 7

El test placebo de fechas aleatorias dejó de ser comparable con la nueva configuración (exposición real 82,5 % frente a 91,2 % en los placebos): las señales reales se agrupan en las temporadas de resultados y el tope de posiciones las recorta, mientras que las fechas aleatorias se reparten. Se añade un aviso automático y el **test B (mismo calendario, otra acción)**, que conserva fechas, duraciones y sorpresas y solo cambia la acción.

## BLOQUE 0: CONFIGURACIÓN DE LA ESTRATEGIA — EARNINGS SURPRISE

In [ ]:
# %%
# ==============================================================================
# BLOQUE 0: CONFIGURACIÓN DE LA ESTRATEGIA — EARNINGS SURPRISE (PEAD) v6
# ==============================================================================

# --- Parámetros financieros ---
CAPITAL_INICIAL      = 10000
COSTE_TRANSACCION    = 0.001       # El motor cobra  coste × rotación UNILATERAL (Σ|Δpeso|/2):
                                   # 0.10% equivale a ~5 pb por lado. Ver el análisis de costes realistas.
FREQ_REBALANCEO      = 'W'         # Revisión semanal ('D' afina el timing si el motor lo admite)
BENCHMARK            = 'SPY'

# --- Rango temporal: se fija con el menú de estrés (Bloque 3) ---
FECHA_INICIO = None
FECHA_FIN    = None

# --- Parámetros de la señal ---
UMBRAL_SORPRESA      = 2.0         # % mínimo de sorpresa de EPS (columna 'Surprise(%)')
SORPRESA_CLIP_PCT    = 50.0        # recorte (EPS cercano a 0 dispara el %)
VENTANA_DRIFT_DIAS   = 60          # SESIONES bursátiles de tenencia desde T+1
FILTRO_REACCION      = True        # exceso de rentabilidad vs SPY en [T-1, T+1] > 0
VOLUMEN_MINIMO_MULT  = 1.5         # pico de volumen (T/T+1) > 1.5x media de 20 sesiones previas
SALIR_ANTES_EARNINGS = True        # no mantener posición durante el siguiente anuncio
MARGEN_PRE_EARNINGS  = 3           # sesiones de margen antes del siguiente anuncio

# --- NUEVO: filtro de momentum PRE-anuncio (tendencia alcista en el cierre de T-1) ---
FILTRO_MOMENTUM        = True
MOMENTUM_MA_CORTA      = 50
MOMENTUM_MA_LARGA      = 200
MOMENTUM_PASA_SIN_DATOS = False    # eventos sin 200 sesiones de historia: False = se rechazan

# --- Gestión del riesgo y tamaño de posición ---
STOP_LOSS_PCT        = -0.08       # desde el precio de entrada; se evalúa en cada decisión
MAX_POSICIONES       = 8           # máximo de posiciones simultáneas
LIMITE_EARNINGS      = 100         # nº de anuncios históricos a pedir a yfinance
# NUEVO: tamaño dinámico por magnitud de sorpresa. Si False: peso fijo 1/MAX_POSICIONES.
TAMANO_DINAMICO      = True
TRAMOS_PESO          = [(10.0, 0.15),   # sorpresa > 10%  -> 15%
                        (5.0,  0.10),   # sorpresa > 5%   -> 10%
                        (0.0,  0.05)]   # resto (> umbral) -> 5%
EXPOSICION_MAXIMA    = 1.0         # si la suma de pesos la supera, se reescala (sin apalancamiento)

# --- NUEVO: costes realistas (mismas unidades que COSTE_TRANSACCION) ---
COSTE_LARGE          = 0.0015      # ≈ 7,5 pb por lado: grandes compañías (spread + deslizamiento)
COSTE_MID            = 0.0040      # ≈ 20 pb por lado: mid-caps (spreads más anchos tras resultados)

# --- NUEVO: universo ---
UNIVERSO_ACTIVO      = 'ambos'     # grupo(s) que generan señales en la ejecución principal:
                                   # 'large' | 'mid' | 'ambos'  (los datos de ambos se cargan siempre)
MOSTRAR_CAP_MERCADO  = True        # muestra la capitalización actual de cada acción (requiere red)

# --- Universo 1: grandes compañías de EE. UU. (el de las versiones anteriores) ---
UNIVERSO_LARGE = {
    'tech_large': {
        'Apple': {'isin': 'AAPL', 'ticker': 'AAPL'},
        'Microsoft': {'isin': 'MSFT', 'ticker': 'MSFT'},
        'Alphabet': {'isin': 'GOOGL', 'ticker': 'GOOGL'},
        'Amazon': {'isin': 'AMZN', 'ticker': 'AMZN'},
        'NVIDIA': {'isin': 'NVDA', 'ticker': 'NVDA'},
        'Meta': {'isin': 'META', 'ticker': 'META'},
        'Tesla': {'isin': 'TSLA', 'ticker': 'TSLA'},
    },
    'finance': {
        'JPMorgan': {'isin': 'JPM', 'ticker': 'JPM'},
        'Visa': {'isin': 'V', 'ticker': 'V'},
        'Bank_of_America': {'isin': 'BAC', 'ticker': 'BAC'},
        'Goldman_Sachs': {'isin': 'GS', 'ticker': 'GS'},
    },
    'healthcare': {
        'Johnson_&_Johnson': {'isin': 'JNJ', 'ticker': 'JNJ'},
        'UnitedHealth': {'isin': 'UNH', 'ticker': 'UNH'},
        'Pfizer': {'isin': 'PFE', 'ticker': 'PFE'},
    },
    'consumer': {
        'Walmart': {'isin': 'WMT', 'ticker': 'WMT'},
        'Procter_&_Gamble': {'isin': 'PG', 'ticker': 'PG'},
        'Coca-Cola': {'isin': 'KO', 'ticker': 'KO'},
        'Nike': {'isin': 'NKE', 'ticker': 'NKE'},
    },
    'industrial': {
        'Boeing': {'isin': 'BA', 'ticker': 'BA'},
        'Caterpillar': {'isin': 'CAT', 'ticker': 'CAT'},
        '3M': {'isin': 'MMM', 'ticker': 'MMM'},
    }
}

# --- Universo 2 (NUEVO): compañías de capitalización media, con cotización anterior a 2008 ---
# OJO: son mid-caps ACTUALES que han sobrevivido hasta hoy → siguen sujetas a sesgo de
# supervivencia. Algunas pueden haber crecido hasta la banda alta o haber cambiado de estatus:
# el Bloque 5 imprime su capitalización actual para que lo verifiques.
UNIVERSO_MID = {
    'tech_mid': {
        'Trimble': {'isin': 'TRMB', 'ticker': 'TRMB'},
        'Akamai': {'isin': 'AKAM', 'ticker': 'AKAM'},
        'Skyworks': {'isin': 'SWKS', 'ticker': 'SWKS'},
        'Zebra_Technologies': {'isin': 'ZBRA', 'ticker': 'ZBRA'},
    },
    'finance_mid': {
        'Raymond_James': {'isin': 'RJF', 'ticker': 'RJF'},
        'Cullen_Frost': {'isin': 'CFR', 'ticker': 'CFR'},
        'Zions': {'isin': 'ZION', 'ticker': 'ZION'},
        'East_West_Bancorp': {'isin': 'EWBC', 'ticker': 'EWBC'},
    },
    'healthcare_mid': {
        'Bio-Rad': {'isin': 'BIO', 'ticker': 'BIO'},
        'Teleflex': {'isin': 'TFX', 'ticker': 'TFX'},
        'Cooper_Companies': {'isin': 'COO', 'ticker': 'COO'},
        'Charles_River': {'isin': 'CRL', 'ticker': 'CRL'},
    },
    'consumer_mid': {
        'Hasbro': {'isin': 'HAS', 'ticker': 'HAS'},
        'Williams-Sonoma': {'isin': 'WSM', 'ticker': 'WSM'},
        'Polaris': {'isin': 'PII', 'ticker': 'PII'},
        'Flowers_Foods': {'isin': 'FLO', 'ticker': 'FLO'},
    },
    'industrial_mid': {
        'Snap-on': {'isin': 'SNA', 'ticker': 'SNA'},
        'Pentair': {'isin': 'PNR', 'ticker': 'PNR'},
        'Hubbell': {'isin': 'HUBB', 'ticker': 'HUBB'},
        'Woodward': {'isin': 'WWD', 'ticker': 'WWD'},
    },
    'materials_energy_mid': {
        'Steel_Dynamics': {'isin': 'STLD', 'ticker': 'STLD'},
        'Reliance_Steel': {'isin': 'RS', 'ticker': 'RS'},
        'Helmerich_&_Payne': {'isin': 'HP', 'ticker': 'HP'},
        'Murphy_Oil': {'isin': 'MUR', 'ticker': 'MUR'},
    },
}

# --- Sector por ticker (para el análisis de subgrupos) ---
SECTORES = {
    'AAPL': 'Tecnología', 'MSFT': 'Tecnología', 'NVDA': 'Tecnología',
    'GOOGL': 'Comunicación', 'META': 'Comunicación',
    'AMZN': 'Consumo discrecional', 'TSLA': 'Consumo discrecional', 'NKE': 'Consumo discrecional',
    'JPM': 'Financiero', 'V': 'Financiero', 'BAC': 'Financiero', 'GS': 'Financiero',
    'JNJ': 'Salud', 'UNH': 'Salud', 'PFE': 'Salud',
    'WMT': 'Consumo básico', 'PG': 'Consumo básico', 'KO': 'Consumo básico',
    'BA': 'Industrial', 'CAT': 'Industrial', 'MMM': 'Industrial',
    'TRMB': 'Tecnología', 'AKAM': 'Tecnología', 'SWKS': 'Tecnología', 'ZBRA': 'Tecnología',
    'RJF': 'Financiero', 'CFR': 'Financiero', 'ZION': 'Financiero', 'EWBC': 'Financiero',
    'BIO': 'Salud', 'TFX': 'Salud', 'COO': 'Salud', 'CRL': 'Salud',
    'HAS': 'Consumo discrecional', 'WSM': 'Consumo discrecional', 'PII': 'Consumo discrecional',
    'FLO': 'Consumo básico',
    'SNA': 'Industrial', 'PNR': 'Industrial', 'HUBB': 'Industrial', 'WWD': 'Industrial',
    'STLD': 'Materiales', 'RS': 'Materiales', 'HP': 'Energía', 'MUR': 'Energía',
}

# Para ampliar el periodo común (Meta empieza en 2012, Tesla en 2010, Visa en 2008) puedes
# excluirlos, p. ej. EXCLUIR_TICKERS = ['META', 'TSLA', 'V'] → el backtest llegaría más atrás (2008)
EXCLUIR_TICKERS = []

def _filtrar(universo):
    f = {c: {n: inf for n, inf in act.items() if inf['ticker'] not in EXCLUIR_TICKERS}
         for c, act in universo.items()}
    return {c: act for c, act in f.items() if act}

UNIVERSO_LARGE = _filtrar(UNIVERSO_LARGE)
UNIVERSO_MID   = _filtrar(UNIVERSO_MID)
UNIVERSO_EARNINGS = {**UNIVERSO_LARGE, **UNIVERSO_MID}      # lo que carga el GestorDatos
GRUPO_TAMANO = {inf['ticker']: g for g, uni in (('large', UNIVERSO_LARGE), ('mid', UNIVERSO_MID))
                for act in uni.values() for inf in act.values()}

n_l = sum(len(a) for a in UNIVERSO_LARGE.values())
n_m = sum(len(a) for a in UNIVERSO_MID.values())
print("✅ Configuración Earnings Surprise v6 cargada:")
print(f"   • Umbral de sorpresa: > {UMBRAL_SORPRESA}% (recorte a {SORPRESA_CLIP_PCT}% para ranking/tamaño)")
print(f"   • Ventana: {VENTANA_DRIFT_DIAS} sesiones desde T+1 | salida pre-anuncio: {SALIR_ANTES_EARNINGS} ({MARGEN_PRE_EARNINGS} ses.)")
print(f"   • Reacción del precio: {FILTRO_REACCION} | volumen > {VOLUMEN_MINIMO_MULT}x | "
      f"momentum MA{MOMENTUM_MA_CORTA}>MA{MOMENTUM_MA_LARGA}: {FILTRO_MOMENTUM}")
if TAMANO_DINAMICO:
    print("   • Tamaño dinámico: " + ", ".join(f"> {m:g}% → {p:.0%}" if m > 0 else f"resto → {p:.0%}" for m, p in TRAMOS_PESO)
          + f" | máx. {MAX_POSICIONES} posiciones, exposición ≤ {EXPOSICION_MAXIMA:.0%}")
else:
    print(f"   • Tamaño fijo: {1/MAX_POSICIONES:.1%} × máx. {MAX_POSICIONES} posiciones")
print(f"   • Stop loss: {STOP_LOSS_PCT:.0%} | costes: base {COSTE_TRANSACCION:.2%}, large {COSTE_LARGE:.2%}, mid {COSTE_MID:.2%}")
print(f"   • Universo cargado: {n_l} grandes + {n_m} mid-caps = {n_l + n_m} acciones | señales sobre: '{UNIVERSO_ACTIVO}'")
if EXCLUIR_TICKERS:
    print(f"   • Excluidos: {EXCLUIR_TICKERS}")

## BLOQUE 1: ENTORNO E IMPORTACIONES

In [2]:
# ==============================================================================
# BLOQUE 1: ENTORNO E IMPORTACIONES
# ==============================================================================
import sys
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')

IN_COLAB = "google.colab" in sys.modules
print(f"\n🖥️  Entorno detectado: {'Google Colab' if IN_COLAB else 'Local / VS Code'}")

def buscar_core():
    try:
        import ipykernel
        try:
            from jupyter_server import serverapp as app_module
        except ImportError:
            from notebook import notebookapp as app_module
        import json, re, urllib.request
        connection_file = os.path.basename(ipykernel.get_connection_file())
        match = re.search(r'kernel-(.*)\.json', connection_file)
        if match:
            kernel_id = match.group(1)
            for srv in app_module.list_running_servers():
                try:
                    url = srv['url'] + 'api/sessions'
                    token = srv.get('token', '')
                    headers = {'Authorization': f'token {token}'} if token else {}
                    req = urllib.request.Request(url, headers=headers)
                    with urllib.request.urlopen(req, timeout=2) as resp:
                        sessions = json.loads(resp.read())
                    for sess in sessions:
                        if sess.get('kernel', {}).get('id') == kernel_id:
                            ruta_nb = os.path.join(srv['root_dir'], sess['notebook']['path'])
                            ruta_dir = os.path.dirname(os.path.abspath(ruta_nb))
                            actual = Path(ruta_dir)
                            for parent in [actual] + list(actual.parents):
                                if (parent / 'core').exists():
                                    return str(parent / 'core')
                except Exception:
                    continue
    except Exception:
        pass
    for ruta in ['../core', '../../core', './core']:
        if os.path.isdir(ruta):
            return os.path.abspath(ruta)
    return None

RUTA_CORE = buscar_core()
if RUTA_CORE and os.path.isdir(RUTA_CORE):
    sys.path.insert(0, os.path.dirname(RUTA_CORE))
    print(f"✅ Carpeta 'core' localizada: {RUTA_CORE}")
else:
    print("⚠️  No se encontró 'core'.")

try:
    from core import (
        GestorProyecto, GestorDatos, MotorBacktestDinamico,
        CalculadorMetricas, CalculadorMetricasExtras, GeneradorInformes
    )
    print("✅ Módulos del motor importados correctamente.")
except ImportError as e:
    print(f"❌ Error importando módulos: {e}")
    raise

BASE_DIR = os.path.dirname(RUTA_CORE) if RUTA_CORE else os.getcwd()
print(f"📂 BASE_DIR del proyecto: {BASE_DIR}")


🖥️  Entorno detectado: Local / VS Code
✅ Carpeta 'core' localizada: /home/enri/Estrategias_No_Convencionales/core
✅ Módulos del motor importados correctamente.
📂 BASE_DIR del proyecto: /home/enri/Estrategias_No_Convencionales


## Bloque 2: Limpieza y Fechas

In [3]:
# %%
# ==============================================================================
# BLOQUE 2: LIMPIEZA DE DIRECTORIOS
# ==============================================================================
gestor_proyecto = GestorProyecto(base_dir_manual=BASE_DIR)
gestor_proyecto.imprimir_resumen()
gestor_proyecto.menu_limpieza_directorios()


📁 BASE_DIR fijado manualmente -> /home/enri/Estrategias_No_Convencionales

🔍 ENTORNO CONFIGURADO
📂 Base: /home/enri/Estrategias_No_Convencionales
📁 CORE                : /home/enri/Estrategias_No_Convencionales/core
📁 CHAPTERS            : /home/enri/Estrategias_No_Convencionales/chapters
📁 LOGS                : /home/enri/Estrategias_No_Convencionales/logs
📁 DATOS               : /home/enri/Estrategias_No_Convencionales/datos
📁 OUTPUTS             : /home/enri/Estrategias_No_Convencionales/outputs
📁 CARTERAS            : /home/enri/Estrategias_No_Convencionales/datos/carteras
📁 DATOS_CSV           : /home/enri/Estrategias_No_Convencionales/datos/datos_csv
📁 DATOS_FI_GESTORA    : /home/enri/Estrategias_No_Convencionales/datos/datos_fi_gestora
📁 DATOS_FI_INVESTING  : /home/enri/Estrategias_No_Convencionales/datos/datos_fi_investing
📁 DATOS_FI_MYINVESTOR : /home/enri/Estrategias_No_Convencionales/datos/datos_fi_myinvestor
📁 DATOS_FI_R4         : /home/enri/Estrategias_No_Convencionales/d


¿Qué directorios deseas VACIAR? (ej: 3,5,7 / 'todos' / 'ninguno'):  ninguno


## BLOQUE 3: GESTIÓN DE FECHAS (MENÚ DE ESTRÉS)

In [ ]:
# %%
# ==============================================================================
# BLOQUE 3: GESTIÓN DE FECHAS (MENÚ DE ESTRÉS)
# ==============================================================================

# --- Inicializar gestor_proyecto si no existe ---
if 'gestor_proyecto' not in locals():
    print("⚠️  gestor_proyecto no inicializado. Creándolo automáticamente...")
    gestor_proyecto = GestorProyecto(base_dir_manual=BASE_DIR)
    gestor_proyecto.imprimir_resumen()

# --- Selección de fechas mediante el menú interactivo de estrés ---
fecha_ini, fecha_fin, nombre_evento = gestor_proyecto.menu_stress_testing()

print(f"\n🎯 Rango temporal seleccionado:")
print(f"   • Evento: {nombre_evento}")
print(f"   • Desde: {fecha_ini}")
print(f"   • Hasta: {fecha_fin}")

# --- Fijar las fechas globales que usa el resto del notebook ---
FECHA_INICIO, FECHA_FIN = fecha_ini, fecha_fin

## BLOQUE 4: MENÚ DEL GESTOR DE DATOS

In [ ]:
# %%
# ==============================================================================
# BLOQUE 4: MENÚ DEL GESTOR DE DATOS
# ==============================================================================
gestor = GestorDatos(
    diccionario_datos=UNIVERSO_EARNINGS,
    fecha_inicio=fecha_ini,
    fecha_fin=fecha_fin,
    gestor_proyecto=gestor_proyecto
)

print(f"\n📊 GestorDatos inicializado con {len(UNIVERSO_EARNINGS)} categorías.")
print("   Ejecutando menú de importación/descarga...\n")

continuar = gestor.menu()

if not continuar:
    print("⛔ Usuario canceló. Abortando notebook.")
else:
    print("\n✅ Datos preparados para el motor.")


📊 GestorDatos inicializado con 5 categorías.
   Ejecutando menú de importación/descarga...


 SISTEMA: IMPORTAR · NORMALIZAR · CARTERAS
1. Descargar/Actualizar cotizaciones (Yahoo)
2. Importar ficheros Renta 4
3. Normalizar CSVs del directorio
4. Crear cartera interactiva (Pide nombre y descarga CSVs)
5. Cargar cartera existente (Lista carteras guardadas)
6. Ejecutar Estrategia (Continuar al Backtest)
7. Salir
8. 🔄 Sincronizar datos con período actual (re-descarga si es necesario)


Selecciona una opción (1-8):  1



🗑️ Limpiando CSVs antiguos y descargando desde Yahoo Finance...

📊 Procesando 21 activos...

   📊 Apple
      ✅ Descargado con ticker: AAPL

   📊 Microsoft
      ✅ Descargado con ticker: MSFT

   📊 Alphabet


$GOOGL: Data doesn't exist for startDate = 951886800, endDate = 1041310800

1 Failed download:
['GOOGL']: Data doesn't exist for startDate = 951886800, endDate = 1041310800
$GOOGL: Data doesn't exist for startDate = 951886800, endDate = 1041310800

1 Failed download:
['GOOGL']: Data doesn't exist for startDate = 951886800, endDate = 1041310800


      ⚠️ No se pudo descargar Alphabet.

   📊 Amazon
      ✅ Descargado con ticker: AMZN

   📊 NVIDIA
      ✅ Descargado con ticker: NVDA

   📊 Meta


...


❌ PROBLEMAS CRÍTICOS detectados:
   • Período solicitado: 2000-03-01 → 2002-12-31
   • Problemas: 4
      • Alphabet: CSV no existe
      • Meta: CSV no existe
      • Tesla: CSV no existe
      • Visa: CSV no existe

⚠️  Los datos en disco no coinciden con el período solicitado.
   Usa la opción 8 para sincronizar, o la opción 1 para re-descargar todo.



Presiona Enter para continuar... 



 SISTEMA: IMPORTAR · NORMALIZAR · CARTERAS
1. Descargar/Actualizar cotizaciones (Yahoo)
2. Importar ficheros Renta 4
3. Normalizar CSVs del directorio
4. Crear cartera interactiva (Pide nombre y descarga CSVs)
5. Cargar cartera existente (Lista carteras guardadas)
6. Ejecutar Estrategia (Continuar al Backtest)
7. Salir
8. 🔄 Sincronizar datos con período actual (re-descarga si es necesario)


Selecciona una opción (1-8):  6



❌ PROBLEMAS CRÍTICOS detectados:
   • Período solicitado: 2000-03-01 → 2002-12-31
   • Problemas: 4
      • Alphabet: CSV no existe
      • Meta: CSV no existe
      • Tesla: CSV no existe
      • Visa: CSV no existe

⚠️  Los datos en disco no coinciden con el período solicitado.
   Usa la opción 8 para sincronizar, o la opción 1 para re-descargar todo.


## BLOQUE 5: GENERADOR DE PESOS DINÁMICO (PEAD)

In [ ]:
# %%
# ==============================================================================
# BLOQUE 5: SEÑALES DE EARNINGS SURPRISE (PEAD) Y GENERADOR DE PESOS — v2
# ==============================================================================
import yfinance as yf

print("\n" + "="*70)
print("⚙️ CONFIGURANDO ESTRATEGIA EARNINGS SURPRISE (PEAD) v6")
print("="*70)

# --- 1. VALIDACIÓN DE DATOS ---
if gestor.precios_ajustados is None or gestor.precios_ajustados.empty:
    raise ValueError("❌ No hay precios ajustados. Ejecuta el Bloque 4 primero.")

precios_sig = gestor.precios_ajustados.copy()
_idx = pd.DatetimeIndex(precios_sig.index)
if _idx.tz is not None:
    _idx = _idx.tz_localize(None)
precios_sig.index = _idx.normalize()
print(f"\n📊 Datos disponibles: {len(precios_sig)} filas "
      f"({precios_sig.index[0].date()} → {precios_sig.index[-1].date()})")

# --- 2. MAPEO columna del DataFrame -> ticker de Yahoo ---
mapa_col_ticker = {}
for categoria, activos in UNIVERSO_EARNINGS.items():
    for nombre, info in activos.items():
        ticker = info['ticker']
        nombre_limpio = str(nombre).replace(" ", "_").replace(".", "").replace(",", "")  # igual que GestorDatos
        if nombre_limpio in precios_sig.columns:
            mapa_col_ticker[nombre_limpio] = ticker
        elif ticker in precios_sig.columns:
            mapa_col_ticker[ticker] = ticker
        else:
            print(f"   ⚠️ {nombre} ({ticker}) no está en los precios cargados; se omite.")

bench_serie = precios_sig[BENCHMARK] if BENCHMARK in precios_sig.columns else None
if bench_serie is None:
    # El motor guarda el benchmark aparte: se descarga aquí para medir la reacción en exceso
    try:
        _b = yf.download(BENCHMARK, start=precios_sig.index[0],
                         end=precios_sig.index[-1] + pd.Timedelta(days=1),
                         auto_adjust=True, progress=False)['Close']
        if isinstance(_b, pd.DataFrame):
            _b = _b.iloc[:, 0]
        _bi = pd.DatetimeIndex(_b.index)
        _b.index = (_bi.tz_localize(None) if _bi.tz is not None else _bi).normalize()
        bench_serie = _b.reindex(precios_sig.index).ffill()
        print(f"   ℹ️ {BENCHMARK} descargado de yfinance para medir la reacción en exceso sobre el mercado.")
    except Exception as e:
        print(f"   ⚠️ No se pudo obtener {BENCHMARK} ({e}): la reacción se medirá SIN ajustar por mercado.")

# --- 2b. GRUPO (large/mid) Y SECTOR de cada columna ---
grupo_col = {col: GRUPO_TAMANO.get(tk, 'large') for col, tk in mapa_col_ticker.items()}
sector_col = {col: SECTORES.get(tk, 'Otros') for col, tk in mapa_col_ticker.items()}
print(f"   Activos mapeados: {sum(g == 'large' for g in grupo_col.values())} grandes + "
      f"{sum(g == 'mid' for g in grupo_col.values())} mid-caps")

if MOSTRAR_CAP_MERCADO:
    caps = {}
    for col, tk in mapa_col_ticker.items():
        try:
            fi = yf.Ticker(tk).fast_info
            caps[col] = getattr(fi, 'market_cap', None) or fi['marketCap']
        except Exception:
            caps[col] = np.nan
    df_caps = pd.DataFrame({'ticker': pd.Series(mapa_col_ticker), 'grupo': pd.Series(grupo_col),
                            'cap_actual_MM_USD': pd.Series(caps) / 1e9}).sort_values(['grupo', 'cap_actual_MM_USD'])
    print("\n   Capitalización ACTUAL (miles de millones USD) — comprueba que las 'mid' lo son:")
    display(df_caps.style.format({'cap_actual_MM_USD': '{:,.1f}'}))

# --- 3. VOLÚMENES (del gestor si los tiene; si no, directamente de yfinance) ---
def cargar_volumenes(mapa, indice):
    vol = pd.DataFrame(np.nan, index=indice, columns=list(mapa))
    g = getattr(gestor, 'volumenes', None)
    if isinstance(g, pd.DataFrame) and not g.empty:
        g = g.copy()
        gi = pd.DatetimeIndex(g.index)
        g.index = (gi.tz_localize(None) if gi.tz is not None else gi).normalize()
        for col in vol.columns:
            for nombre in (f"{col}_Volume", col):
                if nombre in g.columns:
                    vol[col] = g[nombre].reindex(indice)
                    break
    faltan = [c for c in vol.columns if vol[c].isna().all()]
    if faltan:
        try:
            raw = yf.download([mapa[c] for c in faltan], start=indice[0],
                              end=indice[-1] + pd.Timedelta(days=1),
                              auto_adjust=True, progress=False)['Volume']
            if isinstance(raw, pd.Series):
                raw = raw.to_frame(mapa[faltan[0]])
            ri = pd.DatetimeIndex(raw.index)
            raw.index = (ri.tz_localize(None) if ri.tz is not None else ri).normalize()
            for c in faltan:
                if mapa[c] in raw.columns:
                    vol[c] = raw[mapa[c]].reindex(indice)
        except Exception as e:
            print(f"   ⚠️ No se pudieron descargar volúmenes ({e}); el filtro de volumen se omite.")
    return vol

volumenes_sig = cargar_volumenes(mapa_col_ticker, precios_sig.index)
print(f"   Volúmenes disponibles para {int(volumenes_sig.notna().any().sum())}/{len(mapa_col_ticker)} activos")

# --- 4. FECHAS Y SORPRESAS DE EARNINGS ---
def descargar_earnings(ticker, limite):
    tkr = yf.Ticker(ticker)
    try:
        df = tkr.get_earnings_dates(limit=limite)
    except Exception:
        df = tkr.earnings_dates
    if df is None or df.empty:
        return None
    col = next((c for c in df.columns if 'Surprise' in str(c)), None)
    if col is None:
        return None
    out = df[[col]].rename(columns={col: 'sorpresa'}).copy()
    oi = pd.DatetimeIndex(out.index)
    out.index = oi.tz_localize(None) if oi.tz is not None else oi
    return out

print("\n📥 Descargando fechas de earnings...")
earnings_por_activo = {}
for col, ticker in mapa_col_ticker.items():
    try:
        ed = descargar_earnings(ticker, LIMITE_EARNINGS)
        if ed is None:
            print(f"   ⚠️ {ticker}: sin datos de earnings/sorpresa")
            continue
        earnings_por_activo[col] = ed
        validos = ed['sorpresa'].dropna()
        print(f"   {ticker:<6} anuncios con sorpresa: {len(validos):>3}  "
              f"({validos.index.min().date()} → {validos.index.max().date()})")
    except Exception as e:
        print(f"   ❌ {ticker}: {e}")

# ------------------------------------------------------------------------------
# FUNCIONES PURAS (sin dependencias del motor) -> fáciles de testear
# ------------------------------------------------------------------------------
def calcular_tendencia(precios, corta=50, larga=200):
    """DataFrame: 1.0 si MA(corta) > MA(larga) en esa fecha, 0.0 si no, NaN sin historia suficiente."""
    ma_c = precios.rolling(corta).mean()
    ma_l = precios.rolling(larga).mean()
    t = (ma_c > ma_l).astype(float)
    return t.where(ma_c.notna() & ma_l.notna())

def filtro_momentum(precios_hist, ticker):
    """Versión puntual (una acción, última fecha): solo comprar si MA50 > MA200.
    `calcular_tendencia` es su equivalente vectorizado para toda la muestra."""
    ma50 = precios_hist[ticker].rolling(50).mean().iloc[-1]
    ma200 = precios_hist[ticker].rolling(200).mean().iloc[-1]
    return ma50 > ma200


def construir_senales(earnings, precios, volumenes, bench, umbral, ventana,
                      vol_mult, filtro_reaccion, salir_antes, margen_pre, clip_pct,
                      tendencia=None, pasa_sin_datos=False):
    """
    Devuelve (señales, embudo).
      señales : DataFrame (fechas x activos). 0 = sin señal; >0 = sorpresa (%), recortada
                a `clip_pct`, que sirve para ordenar candidatos y para el tamaño de posición.
      embudo  : DataFrame con cuántos eventos sobreviven a cada filtro.

    Convenciones anti look-ahead:
      • La señal empieza el día p0+1 (p0 = primera sesión >= fecha del anuncio).
      • La reacción se mide de cierre(p0-1) a cierre(p0+1), en exceso sobre el benchmark.
      • El momentum PRE-anuncio se evalúa en el cierre de p0-1 (MA corta > MA larga).
      • El volumen del evento (máx. de p0 y p0+1) se compara con la media de las 20
        sesiones PREVIAS (sin incluir el propio evento).
    """
    idx = precios.index
    señales = pd.DataFrame(0.0, index=idx, columns=precios.columns)
    filas = []

    for col, ed in earnings.items():
        if col not in precios.columns:
            continue
        j = precios.columns.get_loc(col)
        px = precios[col].to_numpy(dtype=float)
        vol = (volumenes[col].to_numpy(dtype=float)
               if (volumenes is not None and col in volumenes.columns) else None)
        tend = (tendencia[col].to_numpy(dtype=float)
                if (tendencia is not None and col in tendencia.columns) else None)

        ed = ed.copy()
        ed.index = pd.DatetimeIndex(ed.index).normalize()
        ed = ed[~ed.index.duplicated(keep='last')].sort_index()

        n = dict(activo=col, eventos=0, sorpresa_ok=0, con_precio=0, reaccion_ok=0,
                 momentum_ok=0, volumen_ok=0, sin_historia_ma=0, sin_dato_volumen=0, con_ventana=0)

        for d, s in ed['sorpresa'].items():
            if pd.isna(s):
                continue
            n['eventos'] += 1
            if s <= umbral:
                continue
            n['sorpresa_ok'] += 1

            p0 = idx.searchsorted(d, side='left')
            if p0 < 1 or p0 + 1 >= len(idx):
                continue
            if np.isnan(px[p0 - 1]) or np.isnan(px[p0 + 1]):
                continue
            n['con_precio'] += 1

            # --- Confirmación por reacción del precio (exceso vs benchmark) ---
            if filtro_reaccion:
                r_a = px[p0 + 1] / px[p0 - 1] - 1
                r_b = 0.0
                if bench is not None:
                    b0, b1 = bench.iloc[p0 - 1], bench.iloc[p0 + 1]
                    if not (np.isnan(b0) or np.isnan(b1)):
                        r_b = b1 / b0 - 1
                if (r_a - r_b) <= 0:
                    continue
            n['reaccion_ok'] += 1

            # --- Filtro de momentum PRE-anuncio: tendencia alcista en el cierre de T-1 ---
            if tend is not None:
                tv = tend[p0 - 1]
                if np.isnan(tv):
                    n['sin_historia_ma'] += 1
                    if not pasa_sin_datos:
                        continue
                elif tv < 1.0:
                    continue
            n['momentum_ok'] += 1

            # --- Filtro de volumen anómalo en el evento (un solo test por evento) ---
            if vol is not None and p0 >= 20:
                base = np.nanmean(vol[p0 - 20:p0])
                pico = np.nanmax(vol[p0:p0 + 2]) if not np.all(np.isnan(vol[p0:p0 + 2])) else np.nan
                if np.isnan(base) or base <= 0 or np.isnan(pico):
                    n['sin_dato_volumen'] += 1
                elif pico <= vol_mult * base:
                    continue
            else:
                n['sin_dato_volumen'] += 1
            n['volumen_ok'] += 1

            # --- Ventana de tenencia (en sesiones bursátiles) ---
            entrada = p0 + 1
            fin_excl = entrada + ventana
            if salir_antes:
                proximos = ed.index[ed.index > d + pd.Timedelta(days=5)]
                if len(proximos) and proximos[0] <= idx[-1]:
                    pn = idx.searchsorted(proximos[0], side='left')
                    fin_excl = min(fin_excl, pn - margen_pre)
            fin_excl = min(fin_excl, len(idx))
            if fin_excl <= entrada:
                continue
            n['con_ventana'] += 1

            valor = float(min(s, clip_pct))
            señales.iloc[entrada:fin_excl, j] = np.maximum(
                señales.iloc[entrada:fin_excl, j].to_numpy(), valor)

        filas.append(n)

    embudo = pd.DataFrame(filas).set_index('activo') if filas else pd.DataFrame()
    return señales, embudo


def peso_por_sorpresa(s, tramos):
    """Peso objetivo según la magnitud de la sorpresa. `tramos` = [(sorpresa mínima %, peso), ...]
    ordenado de mayor a menor; se aplica el primer tramo cuya mínima sea superada."""
    for minimo, peso in tramos:
        if s > minimo:
            return peso
    return tramos[-1][1]


def crear_generador_pesos(señales, benchmark, max_posiciones, stop_loss,
                          tamano_dinamico=False, tramos=None, expo_max=1.0):
    """
    Fábrica del generador de pesos para el motor dinámico.
      • Candidatos = activos con señal > 0 (y no expulsados por stop en este episodio).
      • Se ordenan por magnitud de sorpresa y se toman como máximo `max_posiciones`.
      • Tamaño: 1/max_posiciones fijo, o dinámico por tramos de sorpresa (`tramos`).
        Si la suma supera `expo_max`, se reescala (sin apalancamiento).
      • Stop-loss real desde el precio de entrada (se evalúa en cada fecha de decisión).
    """
    peso_unitario = 1.0 / max_posiciones

    def generador(fecha_actual, precios_hist, rent_hist, estado):
        activos = list(precios_hist.columns)
        pesos = {a: 0.0 for a in activos}
        entradas = estado.setdefault('entradas', {})      # activo -> precio de entrada
        bloqueados = estado.setdefault('bloqueados', set())  # expulsados por stop

        fecha = pd.Timestamp(fecha_actual)
        if fecha.tzinfo is not None:
            fecha = fecha.tz_localize(None)
        fecha = fecha.normalize()

        pos = señales.index.searchsorted(fecha, side='right') - 1
        if pos < 0:
            entradas.clear()
            return pesos, estado
        fila = señales.iloc[pos]
        precios_hoy = precios_hist.iloc[-1]

        candidatos = []
        for a in activos:
            if a == benchmark:
                continue
            s = float(fila.get(a, 0.0))
            if s <= 0:                       # episodio terminado: se libera todo
                entradas.pop(a, None)
                bloqueados.discard(a)
                continue
            if a in bloqueados:
                continue
            p = precios_hoy.get(a, np.nan)
            if pd.isna(p):
                continue
            if a in entradas and (p / entradas[a] - 1) <= stop_loss:
                bloqueados.add(a)            # stop-loss: fuera hasta nuevo evento
                entradas.pop(a, None)
                continue
            candidatos.append((a, s, float(p)))

        candidatos.sort(key=lambda x: x[1], reverse=True)
        elegidos = candidatos[:max_posiciones]

        nombres = {a for a, _, _ in elegidos}
        for a in list(entradas):
            if a not in nombres:
                entradas.pop(a, None)
        for a, s, p in elegidos:
            entradas.setdefault(a, p)
            pesos[a] = peso_por_sorpresa(s, tramos) if (tamano_dinamico and tramos) else peso_unitario

        total = sum(pesos.values())
        if total > expo_max:                 # sin apalancamiento
            pesos = {a: w * expo_max / total for a, w in pesos.items()}

        estado.setdefault('hist_exposicion', {})[fecha] = sum(pesos.values())
        return pesos, estado

    return generador


# --- 5. TENDENCIA PRE-ANUNCIO Y GRUPOS ---
tendencia_sig = calcular_tendencia(precios_sig, MOMENTUM_MA_CORTA, MOMENTUM_MA_LARGA)

def columnas_del_universo(universo):
    if universo == 'ambos':
        return list(mapa_col_ticker)
    return [c for c in mapa_col_ticker if grupo_col[c] == universo]

def generar_senales(umbral=None, ventana=None, momentum=None, universo=None, devolver_embudo=False):
    """Construye las señales con los parámetros globales; cualquiera puede sobrescribirse
    (lo usan el barrido, las variantes y los subgrupos para que todo pase por el mismo código)."""
    umbral = UMBRAL_SORPRESA if umbral is None else umbral
    ventana = VENTANA_DRIFT_DIAS if ventana is None else ventana
    momentum = FILTRO_MOMENTUM if momentum is None else momentum
    universo = UNIVERSO_ACTIVO if universo is None else universo
    earn = {c: earnings_por_activo[c] for c in columnas_del_universo(universo) if c in earnings_por_activo}
    sen, emb = construir_senales(
        earn, precios_sig, volumenes_sig, bench_serie,
        umbral=umbral, ventana=ventana, vol_mult=VOLUMEN_MINIMO_MULT,
        filtro_reaccion=FILTRO_REACCION, salir_antes=SALIR_ANTES_EARNINGS,
        margen_pre=MARGEN_PRE_EARNINGS, clip_pct=SORPRESA_CLIP_PCT,
        tendencia=tendencia_sig if momentum else None,
        pasa_sin_datos=MOMENTUM_PASA_SIN_DATOS)
    return (sen, emb) if devolver_embudo else sen

# --- 6. CONSTRUCCIÓN DE SEÑALES (configuración principal) ---
señales_finales, embudo = generar_senales(devolver_embudo=True)

n_activas = (señales_finales.drop(columns=[BENCHMARK], errors='ignore') > 0).sum(axis=1)
print("\n" + "="*70)
print(f"📊 EMBUDO DE SEÑALES — universo '{UNIVERSO_ACTIVO}', momentum {'ON' if FILTRO_MOMENTUM else 'OFF'}")
print("="*70)
if not embudo.empty:
    display(embudo)
    tot = embudo.sum()
    print(f"\n   Anuncios analizados: {tot['eventos']:.0f} → sorpresa > umbral: {tot['sorpresa_ok']:.0f} "
          f"→ reacción OK: {tot['reaccion_ok']:.0f} → momentum OK: {tot['momentum_ok']:.0f} "
          f"→ volumen OK: {tot['volumen_ok']:.0f} → con ventana: {tot['con_ventana']:.0f}")
    if tot['sin_historia_ma'] > 0:
        print(f"   ℹ️ {tot['sin_historia_ma']:.0f} eventos sin {MOMENTUM_MA_LARGA} sesiones de historia para la media larga "
              f"({'rechazados' if not MOMENTUM_PASA_SIN_DATOS else 'aceptados'}).")
    if tot['sin_dato_volumen'] > 0:
        print(f"   ⚠️ {tot['sin_dato_volumen']:.0f} eventos sin dato de volumen (pasaron sin filtrar).")
    por_grupo = embudo.groupby(embudo.index.map(grupo_col))[['eventos', 'sorpresa_ok', 'reaccion_ok', 'momentum_ok', 'con_ventana']].sum()
    print("\n   Por tamaño de compañía:")
    display(por_grupo)
print(f"   Días con ≥1 señal: {(n_activas > 0).sum()} ({(n_activas > 0).mean():.1%} de la muestra)")
print(f"   Nº medio de señales simultáneas (días con señal): {n_activas[n_activas > 0].mean():.1f}")
if (n_activas > 0).sum() == 0:
    print("   ⚠️ Sin señales en este rango: amplía las fechas o relaja los filtros.")
print("="*70)

# --- 7. GENERADOR DE PESOS ---
generador_pesos_earnings = crear_generador_pesos(
    señales_finales, BENCHMARK, MAX_POSICIONES, STOP_LOSS_PCT,
    tamano_dinamico=TAMANO_DINAMICO, tramos=TRAMOS_PESO, expo_max=EXPOSICION_MAXIMA
)
print("\n✅ Generador de pesos PEAD v6 configurado.")

## BLOQUE 6: EJECUCIÓN DEL BACKTEST DINÁMICO

In [ ]:
# %%
# ==============================================================================
# BLOQUE 6: EJECUCIÓN DEL MOTOR DE BACKTEST DINÁMICO
# ==============================================================================

print("\n" + "="*70)
print("⏳ EJECUTANDO BACKTEST EARNINGS SURPRISE (MEJORADA)")
print("="*70)

motor_dinamico = MotorBacktestDinamico(
    precios=gestor.precios_ajustados,
    rentabilidades=gestor.rentabilidades_diarias,
    generador_pesos=generador_pesos_earnings,
    frecuencia_decision=FREQ_REBALANCEO,
    coste_transaccion=COSTE_TRANSACCION,
    benchmark_ticker=BENCHMARK,
    permitir_short=False,
    max_short_porcentaje=0.0
)

serie_patrimonio = motor_dinamico.ejecutar_backtest(
    capital_inicial=CAPITAL_INICIAL,
    estado_inicial={}
)

print("\n✅ Backtest completado.")
print(f"   • Patrimonio final: {serie_patrimonio.iloc[-1]:,.2f} €")
print(f"   • Rentabilidad total: {((serie_patrimonio.iloc[-1] / CAPITAL_INICIAL) - 1) * 100:.2f}%")
print(f"   • Transacciones de rebalanceo: {len(motor_dinamico.log_transacciones)}")

## BLOQUE 7: MÉTRICAS E INFORMES

In [ ]:
# %%
# ==============================================================================
# BLOQUE 7: MÉTRICAS E INFORMES
# ==============================================================================

auditor = CalculadorMetricas(
    serie_patrimonio=motor_dinamico.serie_patrimonio,
    serie_buy_hold=motor_dinamico.serie_buy_hold,
    serie_benchmark=motor_dinamico.serie_benchmark
)
metricas_reb, metricas_bh = auditor.calcular_todas_las_metricas(rf=0.02)

metricas_extras_calc = CalculadorMetricasExtras(
    serie_patrimonio=motor_dinamico.serie_patrimonio,
    log_operaciones=motor_dinamico.log_operaciones,
    serie_benchmark=motor_dinamico.serie_benchmark
)
metricas_extras = metricas_extras_calc.calcular_todas_las_metricas()

informe = GeneradorInformes(
    serie_patrimonio=motor_dinamico.serie_patrimonio,
    metricas=(metricas_reb, metricas_bh),
    nombre_estrategia="Earnings Surprise (PEAD)",
    serie_buy_hold=motor_dinamico.serie_buy_hold,
    log_transacciones=motor_dinamico.log_transacciones,
    pesos_actuales=motor_dinamico.pesos_actuales_finales,
    pesos_ideales=None
)

informe.imprimir_metricas()
print("\n🩺 DIAGNÓSTICO ESTÁNDAR:")
print("  " + auditor.diagnosticar())

print("\n" + "="*70)
print("📜 LOG DE REBALANCEOS (Últimas 15 operaciones)")
print("="*70)
if len(motor_dinamico.log_transacciones) > 0:
    df_log = pd.DataFrame(motor_dinamico.log_transacciones).tail(15)
    display(df_log[['Fecha', 'Rotacion %', 'Coste (€)', 'Patrimonio (€)']])
else:
    print("ℹ️ No se registraron rebalanceos.")

print("\n📐 MÉTRICAS EXTRA:")
if isinstance(metricas_extras, dict):
    for k, v in metricas_extras.items():
        print(f"   • {k}: {v}")
else:
    print(metricas_extras)

## BLOQUE 8: VISUALIZACIÓN Y GUARDADO DE GRÁFICOS

In [ ]:
# %%
# ==============================================================================
# BLOQUE 8: VISUALIZACIÓN Y GUARDADO DE GRÁFICOS
# ==============================================================================

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except Exception:
    plt.style.use('seaborn-darkgrid')

pat = motor_dinamico.serie_patrimonio
bh  = motor_dinamico.serie_buy_hold
ben = motor_dinamico.serie_benchmark

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(pat.index, pat, label='Earnings Surprise', color='tab:purple', lw=2.2)
ax.plot(bh.index, bh, label='Buy & Hold (Universo)', color='tab:gray', lw=1.5, ls='--')
if ben is not None and not ben.empty:
    # Mismo eje y misma base: las curvas son directamente comparables
    ax.plot(ben.index, ben / ben.iloc[0] * CAPITAL_INICIAL,
            label=f'{BENCHMARK} (normalizado)', color='tab:red', lw=1.5, ls='-.')
ax.set_xlabel('Fecha', fontsize=11)
ax.set_ylabel('Patrimonio (€)', fontsize=11)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)

titulo = f'Earnings Surprise (PEAD) — {pat.index[0].year} a {pat.index[-1].year}'
plt.title(titulo, fontsize=13, fontweight='bold', pad=15)
fig.tight_layout()

# Guardar ANTES de mostrar (algunos backends cierran la figura al hacer show)
nombre_archivo = "09_earnings_surprise_backtest"
ruta_guardado = Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / f"{nombre_archivo}.png"
ruta_guardado.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ruta_guardado, dpi=150, bbox_inches='tight')
print(f"\n✅ Gráfico guardado en: {ruta_guardado}")
plt.show()
plt.close(fig)

## Análisis de resultados

In [ ]:
# %%
# ==============================================================================
# ANÁLISIS DE RESULTADOS Y COMPARATIVA
# ==============================================================================

print("\n" + "="*70)
print("📊 ANÁLISIS DE RESULTADOS")
print("="*70)

pat = motor_dinamico.serie_patrimonio
bh  = motor_dinamico.serie_buy_hold
ben = motor_dinamico.serie_benchmark

anios = (pat.index[-1] - pat.index[0]).days / 365.25
def _cagr(final, inicial):
    return (final / inicial) ** (1 / anios) - 1 if anios > 0 else np.nan

rent_total = pat.iloc[-1] / CAPITAL_INICIAL - 1
n_sim = (señales_finales.drop(columns=[BENCHMARK], errors='ignore') > 0).sum(axis=1)
dias_senal = int((n_sim > 0).sum())

print(f"\n📈 MÉTRICAS CLAVE ({anios:.1f} años):")
print(f"   • Rentabilidad total estrategia: {rent_total:.2%} | CAGR: {_cagr(pat.iloc[-1], CAPITAL_INICIAL):.2%}")
print(f"   • Buy & Hold universo:           {bh.iloc[-1] / bh.iloc[0] - 1:.2%} | CAGR: {_cagr(bh.iloc[-1], bh.iloc[0]):.2%}")
if ben is not None and not ben.empty:
    print(f"   • {BENCHMARK}:                          {ben.iloc[-1] / ben.iloc[0] - 1:.2%} | CAGR: {_cagr(ben.iloc[-1], ben.iloc[0]):.2%}")
print(f"   • Días con ≥1 señal: {dias_senal} ({dias_senal / len(n_sim):.1%}) "
      f"— NO es la exposición; mira la exposición real justo debajo")
_h = getattr(motor_dinamico, 'estado_final', {}).get('hist_exposicion', {})
if _h:
    print(f"   • Exposición REAL media (suma de pesos objetivo): {np.mean(list(_h.values())):.1%} "
          f"(mín {np.min(list(_h.values())):.0%}, máx {np.max(list(_h.values())):.0%})")
print(f"   • Nº de rebalanceos: {len(motor_dinamico.log_transacciones)}")

if len(motor_dinamico.log_transacciones) > 0:
    df_log = pd.DataFrame(motor_dinamico.log_transacciones)
    print(f"\n ÚLTIMOS 10 REBALANCEOS:")
    print(df_log[['Fecha', 'Rotacion %', 'Coste (€)', 'Patrimonio (€)']].tail(10).to_string(index=False))

print("\n💡 Lectura: con exposición parcial, compara también la rentabilidad por unidad de "
      "riesgo (Sharpe, drawdown) y no solo el total frente a SPY.")
print("\n" + "="*70)
print("✅ ANÁLISIS COMPLETADO")
print("="*70)

## ESTADÍSTICAS POR EVENTO

In [ ]:
# %%
# ==============================================================================
# ESTADÍSTICAS POR EVENTO (sustituyen a las "métricas extra" cuando el motor no
# registra operaciones: Win-rate = 0 y Operaciones = 0 indican log vacío)
# ==============================================================================
def estadisticas_por_evento(señales, precios, bench, stop_loss):
    """Rentabilidad de cada episodio de señal (cierre del primer día -> cierre del último),
    en bruto y en exceso sobre el benchmark. Aproximación: no incluye costes ni el timing
    exacto del motor; el stop-loss se marca si el mínimo intra-episodio lo habría activado."""
    filas = []
    for col in señales.columns:
        if col not in precios.columns or col == BENCHMARK:
            continue
        activa = (señales[col].to_numpy() > 0).astype(int)
        dif = np.diff(np.r_[0, activa, 0])
        for i, f in zip(np.where(dif == 1)[0], np.where(dif == -1)[0] - 1):
            tramo = precios[col].iloc[i:f + 1]
            p_in, p_out = tramo.iloc[0], tramo.iloc[-1]
            if pd.isna(p_in) or pd.isna(p_out) or p_in <= 0:
                continue
            r = p_out / p_in - 1
            r_b = np.nan
            if bench is not None and not np.isnan(bench.iloc[i]) and not np.isnan(bench.iloc[f]):
                r_b = bench.iloc[f] / bench.iloc[i] - 1
            tocado = (tramo.min() / p_in - 1) <= stop_loss
            filas.append(dict(activo=col, entrada=precios.index[i], salida=precios.index[f],
                              sesiones=f - i + 1, rent=r, rent_bench=r_b, exceso=r - r_b,
                              rent_con_stop=(stop_loss if tocado else r), stop=tocado))
    return pd.DataFrame(filas)


ev = estadisticas_por_evento(señales_finales, precios_sig, bench_serie, STOP_LOSS_PCT)
if ev.empty:
    print("ℹ️ Sin episodios de señal en el rango.")
else:
    n = len(ev)
    ex = ev['exceso'].dropna()
    t_est = ex.mean() / (ex.std(ddof=1) / np.sqrt(len(ex))) if len(ex) > 2 else np.nan
    print("="*70)
    print("📊 ESTADÍSTICAS POR EVENTO (cierre de entrada → cierre de salida, sin costes)")
    print("="*70)
    print(f"   • Episodios: {n} | duración media: {ev['sesiones'].mean():.0f} sesiones")
    print(f"   • Acierto (rent > 0):            {(ev['rent'] > 0).mean():.1%}")
    print(f"   • Rentabilidad media / mediana:  {ev['rent'].mean():.2%} / {ev['rent'].median():.2%}")
    print(f"   • Exceso medio vs {BENCHMARK}:        {ex.mean():.2%} | acierto vs {BENCHMARK}: {(ex > 0).mean():.1%}")
    print(f"   • Episodios que tocaron el stop: {ev['stop'].mean():.1%} "
          f"(rent. media aplicando stop: {ev['rent_con_stop'].mean():.2%})")
    print(f"   • t-stat del exceso: {t_est:.2f}  (orientativo: los eventos se agrupan en las mismas "
          f"temporadas de resultados, así que sobrestima la significación)")
    display(ev.groupby(ev['entrada'].dt.year)[['rent', 'exceso']].mean().style.format('{:.2%}'))

## ESTUDIO DE EVENTOS: ¿hay drift más allá del que ya tienen estas acciones?

In [ ]:
# %%
# ==============================================================================
# ESTUDIO DE EVENTOS: exceso acumulado tras la señal vs el exceso "normal" de cada acción
# ==============================================================================
# Comparar contra SPY mezcla el efecto PEAD con el hecho de que estas 21 acciones (elegidas
# con información de hoy) han batido a SPY. Aquí cada evento se compara con el exceso medio
# INCONDICIONAL de la misma acción a ese mismo horizonte: eso aísla el efecto de la señal.
HORIZONTES = (5, 10, 20, 40, 60)

def estudio_de_eventos(señales, precios, bench, horizontes=HORIZONTES):
    px = precios.to_numpy(dtype=float)
    b = bench.to_numpy(dtype=float)
    filas = []
    for h in horizontes:
        dif_ev, exc_ev = [], []
        for j, col in enumerate(precios.columns):
            if col == BENCHMARK or col not in señales.columns:
                continue
            ex_all = (px[h:, j] / px[:-h, j] - 1) - (b[h:] / b[:-h] - 1)
            base = np.nanmean(ex_all)
            act = np.diff(np.r_[0, (señales[col].to_numpy() > 0).astype(int), 0])
            for i in np.where(act == 1)[0]:
                if i + h < len(px) and not np.isnan(px[i, j]) and not np.isnan(px[i + h, j]):
                    e = (px[i + h, j] / px[i, j] - 1) - (b[i + h] / b[i] - 1)
                    exc_ev.append(e)
                    dif_ev.append(e - base)
        dif_ev = np.array(dif_ev)
        t = dif_ev.mean() / (dif_ev.std(ddof=1) / np.sqrt(len(dif_ev))) if len(dif_ev) > 2 else np.nan
        filas.append(dict(horizonte=h, eventos=len(dif_ev), exceso_evento=np.mean(exc_ev),
                          exceso_normal=np.mean(exc_ev) - dif_ev.mean(), exceso_extra=dif_ev.mean(), t=t))
    return pd.DataFrame(filas).set_index('horizonte')

if bench_serie is None:
    print("⚠️ Sin serie de benchmark: no se puede hacer el estudio de eventos.")
else:
    est = estudio_de_eventos(señales_finales, precios_sig, bench_serie.reindex(precios_sig.index).ffill())
    print("="*70)
    print("📊 ESTUDIO DE EVENTOS (exceso sobre SPY, desde el cierre del día de entrada)")
    print("="*70)
    display(est.style.format({'exceso_evento': '{:.2%}', 'exceso_normal': '{:.2%}',
                              'exceso_extra': '{:.2%}', 't': '{:.2f}'}))
    print("   exceso_extra = exceso del evento − exceso medio de esa acción en cualquier fecha.")
    print("   Si hay drift PEAD, 'exceso_extra' debe ser positivo y crecer sobre todo en los primeros horizontes.")
    print("   (t-stat orientativo: los eventos se agrupan en las mismas temporadas de resultados.)")

def detalle_eventos(señales, precios, bench, horizontes=(20, 60)):
    """Una fila por episodio de señal: exceso sobre el benchmark a cada horizonte (`exc_h`) y
    exceso EXTRA sobre el normal de esa acción (`extra_h`). Misma lógica que estudio_de_eventos."""
    px = precios.to_numpy(dtype=float)
    b = bench.to_numpy(dtype=float)
    filas = []
    for j, col in enumerate(precios.columns):
        if col == BENCHMARK or col not in señales.columns:
            continue
        v = señales[col].to_numpy()
        act = np.diff(np.r_[0, (v > 0).astype(int), 0])
        inicios = np.where(act == 1)[0]
        if len(inicios) == 0:
            continue
        base = {h: np.nanmean((px[h:, j] / px[:-h, j] - 1) - (b[h:] / b[:-h] - 1)) for h in horizontes}
        for i in inicios:
            fila = dict(activo=col, i=int(i), fecha=precios.index[i], sorpresa=float(v[i]))
            for h in horizontes:
                if i + h < len(px) and not np.isnan(px[i, j]) and not np.isnan(px[i + h, j]):
                    e = (px[i + h, j] / px[i, j] - 1) - (b[i + h] / b[i] - 1)
                    fila[f'exc_{h}'], fila[f'extra_{h}'] = e, e - base[h]
                else:
                    fila[f'exc_{h}'], fila[f'extra_{h}'] = np.nan, np.nan
            filas.append(fila)
    return pd.DataFrame(filas)

## TEST PLACEBO: ¿aporta algo la señal?

In [ ]:
# %%
# ==============================================================================
# TEST PLACEBO: ¿aporta algo la señal de earnings o todo viene del universo?
# ==============================================================================
# Se conservan, para cada acción, el nº y la duración de los episodios de la estrategia
# real (misma exposición por activo), pero se colocan en fechas ALEATORIAS. Si la
# estrategia real no supera claramente a esa distribución, el timing de los earnings
# no aporta información y el resultado viene de estar invertido en estas acciones.
import time

N_PLACEBOS = 100
SEMILLA    = 42

def _resumen_serie(s, rf=0.02):
    """Mismas convenciones que CalculadorMetricas: Sharpe = (CAGR - rf) / volatilidad."""
    s = s.dropna()
    r = s.pct_change().dropna()
    anios = (s.index[-1] - s.index[0]).days / 365.25
    cagr = (s.iloc[-1] / s.iloc[0]) ** (1 / anios) - 1
    vol = r.std() * np.sqrt(252)
    return dict(CAGR=cagr, Vol=vol, Sharpe=(cagr - rf) / vol if vol > 0 else np.nan,
                MaxDD=((s - s.cummax()) / s.cummax()).min())

def _backtest_rapido(señales, max_pos, tamano=None, coste=None, devolver_motor=False):
    """Backtest con el motor real, sin descargar benchmark. `tamano`/`coste` sobrescriben la configuración."""
    tamano = TAMANO_DINAMICO if tamano is None else tamano
    gen = crear_generador_pesos(señales, BENCHMARK, max_pos, STOP_LOSS_PCT,
                                tamano_dinamico=tamano, tramos=TRAMOS_PESO, expo_max=EXPOSICION_MAXIMA)
    motor = MotorBacktestDinamico(
        precios=gestor.precios_ajustados, rentabilidades=gestor.rentabilidades_diarias,
        generador_pesos=gen, frecuencia_decision=FREQ_REBALANCEO,
        coste_transaccion=COSTE_TRANSACCION if coste is None else coste,
        benchmark_ticker=None,   # sin descargas
        permitir_short=False, max_short_porcentaje=0.0)
    serie = motor.ejecutar_backtest(capital_inicial=CAPITAL_INICIAL, estado_inicial={})
    return (serie, motor) if devolver_motor else serie

def placebo_senales(señales, rng, intentos=50):
    """Reubica cada episodio en una fecha aleatoria SIN solaparse con otros episodios del
    mismo activo (así el placebo conserva la exposición por activo de la estrategia real;
    si se permitieran solapes, el placebo estaría menos invertido y sesgaría el test)."""
    v = señales.to_numpy()
    n = len(v)
    out = np.zeros_like(v)
    for j in range(v.shape[1]):
        ocupado = np.zeros(n, dtype=bool)
        dif = np.diff(np.r_[0, (v[:, j] > 0).astype(int), 0])
        for i, f in zip(np.where(dif == 1)[0], np.where(dif == -1)[0] - 1):
            L = f - i + 1
            for _ in range(intentos):
                s0 = int(rng.integers(0, n - L + 1))
                if not ocupado[s0:s0 + L].any():
                    break
            ocupado[s0:s0 + L] = True
            out[s0:s0 + L, j] = np.maximum(out[s0:s0 + L, j], v[i, j])
    return pd.DataFrame(out, index=señales.index, columns=señales.columns)

def _expo_senales(sen, m, tamano=None):
    """Exposición teórica media según señales (cota superior: los stops dejan más efectivo)."""
    tamano = TAMANO_DINAMICO if tamano is None else tamano
    a = sen.drop(columns=[BENCHMARK], errors='ignore').to_numpy()
    top = -np.sort(-a, axis=1)[:, :m]
    if tamano:
        w = np.select([top > t for t, _ in TRAMOS_PESO], [p for _, p in TRAMOS_PESO], default=TRAMOS_PESO[-1][1])
        w = np.where(top > 0, w, 0.0)
    else:
        w = np.where(top > 0, 1.0 / m, 0.0)
    return float(np.minimum(w.sum(axis=1), EXPOSICION_MAXIMA).mean())

t0 = time.time()
real = _resumen_serie(_backtest_rapido(señales_finales, MAX_POSICIONES))
expo_real = _expo_senales(señales_finales, MAX_POSICIONES)
t1 = time.time() - t0
print(f"Estrategia real: CAGR {real['CAGR']:.2%} | Sharpe {real['Sharpe']:.2f} | MaxDD {real['MaxDD']:.1%}")
print(f"Un backtest tarda ~{t1:.1f}s → {N_PLACEBOS} placebos ≈ {N_PLACEBOS * t1 / 60:.1f} min\n")

rng = np.random.default_rng(SEMILLA)
filas = []
for k in range(N_PLACEBOS):
    sen_p = placebo_senales(señales_finales, rng)
    r_p = _resumen_serie(_backtest_rapido(sen_p, MAX_POSICIONES))
    r_p['expo'] = _expo_senales(sen_p, MAX_POSICIONES)
    filas.append(r_p)
    if (k + 1) % 10 == 0:
        print(f"   placebo {k + 1}/{N_PLACEBOS}")
df_pl = pd.DataFrame(filas)

print("\n" + "="*70)
print("📊 RESULTADO DEL TEST PLACEBO")
print("="*70)
print(f"   Exposición por señales: real {expo_real:.1%} | placebos {df_pl['expo'].mean():.1%} "
      f"(deben ser casi iguales; si no, el test no es comparable)")
resumen = {}
for m, fmt in (('CAGR', '{:.2%}'), ('Sharpe', '{:.2f}')):
    p = (1 + (df_pl[m] >= real[m]).sum()) / (1 + len(df_pl))
    resumen[m] = p
    print(f"   {m:<7} real {fmt.format(real[m])} | placebos: media {fmt.format(df_pl[m].mean())}, "
          f"p5 {fmt.format(df_pl[m].quantile(.05))}, p95 {fmt.format(df_pl[m].quantile(.95))} "
          f"| p-valor (unilateral) = {p:.3f}")
p_ref = resumen['Sharpe']   # el Sharpe es menos sensible a diferencias de exposición que el CAGR
if abs(expo_real - df_pl['expo'].mean()) > 0.03:
    print(f"\n   ⚠️ NO COMPARABLE: los placebos están {df_pl['expo'].mean() - expo_real:+.1%} más invertidos que la estrategia real.\n"
          "      Las señales reales se agrupan en las temporadas de resultados y el tope de posiciones las recorta;\n"
          "      las fechas aleatorias se reparten y no. Interpreta SOLO el test B (mismo calendario, otra acción).")
elif p_ref < 0.05:
    print("\n   ✅ La estrategia real supera a la gran mayoría de placebos: el timing de los earnings aporta algo.")
elif p_ref < 0.15:
    print("\n   🟡 Evidencia DÉBIL: la estrategia real queda en la cola alta de los placebos, pero sin llegar\n"
          "      a ser concluyente (p entre 0.05 y 0.15). Con 100 placebos y un solo periodo, no basta para afirmar un efecto.")
else:
    print("\n   ⚠️ La estrategia real NO se distingue de elegir fechas al azar con la misma exposición:\n"
          "      no hay evidencia de que la señal PEAD añada valor sobre el universo.")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, m, fmt in zip(axs, ('CAGR', 'Sharpe'), ('{:.0%}', '{:.1f}')):
    ax.hist(df_pl[m], bins=20, color='tab:gray', alpha=.7)
    ax.axvline(real[m], color='tab:purple', lw=2.5, label='Estrategia real')
    ax.set_title(f'{m}: real vs {N_PLACEBOS} placebos (p={resumen[m]:.3f})')
    ax.legend()
fig.tight_layout()
ruta_pl = Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / "09_earnings_surprise_placebo.png"
ruta_pl.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ruta_pl, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

# ==============================================================================
# TEST B: MISMO CALENDARIO, OTRA ACCIÓN
# ==============================================================================
# Cada episodio conserva su fecha de inicio, su duración y su sorpresa (por tanto el
# calendario, el agrupamiento por temporadas, el tope de posiciones y la exposición), pero se
# asigna a OTRA acción del universo elegida al azar. Mide si la regla de selección (sorpresa +
# reacción + momentum + volumen) escoge mejores acciones que el azar en esas mismas fechas.
def placebo_por_activo(señales, rng, cols_ok, intentos=50):
    v = señales.to_numpy()
    cols = list(señales.columns)
    idx_ok = [cols.index(c) for c in cols_ok if c in cols]
    out = np.zeros_like(v)
    ocup = np.zeros(v.shape, dtype=bool)
    for j in idx_ok:
        dif = np.diff(np.r_[0, (v[:, j] > 0).astype(int), 0])
        for i, f in zip(np.where(dif == 1)[0], np.where(dif == -1)[0] - 1):
            elegido = j                                   # si no hay hueco, se deja en su sitio
            for _ in range(intentos):
                j2 = int(rng.choice(idx_ok))
                if not ocup[i:f + 1, j2].any():
                    elegido = j2
                    break
            ocup[i:f + 1, elegido] = True
            out[i:f + 1, elegido] = np.maximum(out[i:f + 1, elegido], v[i, j])
    return pd.DataFrame(out, index=señales.index, columns=señales.columns)

cols_ok = columnas_del_universo(UNIVERSO_ACTIVO)
rng_b = np.random.default_rng(SEMILLA + 1)
filas_b = []
for k in range(N_PLACEBOS):
    sen_b = placebo_por_activo(señales_finales, rng_b, cols_ok)
    r_b = _resumen_serie(_backtest_rapido(sen_b, MAX_POSICIONES))
    r_b['expo'] = _expo_senales(sen_b, MAX_POSICIONES)
    filas_b.append(r_b)
    if (k + 1) % 10 == 0:
        print(f"   placebo B {k + 1}/{N_PLACEBOS}")
df_pl_b = pd.DataFrame(filas_b)

print("\n" + "="*70)
print("📊 TEST B — MISMO CALENDARIO, OTRA ACCIÓN")
print("="*70)
print(f"   Exposición por señales: real {expo_real:.1%} | placebos {df_pl_b['expo'].mean():.1%}")
resumen_b = {}
for m, fmt in (('CAGR', '{:.2%}'), ('Sharpe', '{:.2f}')):
    p = (1 + (df_pl_b[m] >= real[m]).sum()) / (1 + len(df_pl_b))
    resumen_b[m] = p
    print(f"   {m:<7} real {fmt.format(real[m])} | placebos: media {fmt.format(df_pl_b[m].mean())}, "
          f"p5 {fmt.format(df_pl_b[m].quantile(.05))}, p95 {fmt.format(df_pl_b[m].quantile(.95))} "
          f"| p-valor (unilateral) = {p:.3f}")
p_b = resumen_b['Sharpe']
if abs(expo_real - df_pl_b['expo'].mean()) > 0.03:
    print("\n   ⚠️ Aun así la exposición no coincide: interpreta con cautela.")
elif p_b < 0.05:
    print("\n   ✅ La regla de selección supera a elegir acciones al azar en las mismas fechas.")
elif p_b < 0.15:
    print("\n   🟡 Evidencia DÉBIL: la selección queda en la cola alta, sin ser concluyente.")
else:
    print("\n   ⚠️ La selección de acciones NO se distingue de elegir acciones al azar en las mismas fechas.")

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axs, ('CAGR', 'Sharpe')):
    ax.hist(df_pl_b[m], bins=20, color='tab:gray', alpha=.7)
    ax.axvline(real[m], color='tab:purple', lw=2.5, label='Estrategia real')
    ax.set_title(f'Test B — {m}: real vs {N_PLACEBOS} placebos (p={resumen_b[m]:.3f})')
    ax.legend()
fig.tight_layout()
fig.savefig(Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / "09_earnings_surprise_placebo_B.png", dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## BARRIDO DE PARÁMETROS: robustez

In [ ]:
# %%
# ==============================================================================
# BARRIDO DE PARÁMETROS: ¿es robusto el resultado o depende de un punto concreto?
# ==============================================================================
# OJO: probar muchas combinaciones sobre los mismos datos y quedarse con la mejor es
# data snooping. Aquí se usa para medir la ESTABILIDAD (dispersión), no para elegir.
from itertools import product

GRID_UMBRAL = [2, 5, 10]
GRID_VENTANA = [20, 40, 60]
GRID_MAXPOS = [5, 8, 12]

filas = []
total = len(GRID_UMBRAL) * len(GRID_VENTANA) * len(GRID_MAXPOS)
for k, (u, v, m) in enumerate(product(GRID_UMBRAL, GRID_VENTANA, GRID_MAXPOS), 1):
    sen = generar_senales(umbral=u, ventana=v)
    r = _resumen_serie(_backtest_rapido(sen, m))
    r.update(umbral=u, ventana=v, max_pos=m, expo=_expo_senales(sen, m))
    filas.append(r)
    if k % 9 == 0:
        print(f"   {k}/{total} combinaciones")
df_sw = pd.DataFrame(filas)[['umbral', 'ventana', 'max_pos', 'CAGR', 'Vol', 'Sharpe', 'MaxDD', 'expo']]

ref = {'SPY': _resumen_serie(bench_serie.dropna()),
       'Buy & Hold universo': _resumen_serie(motor_dinamico.serie_buy_hold)}
print("\n" + "="*70)
print("📊 BARRIDO DE PARÁMETROS (ordenado por Sharpe)")
print("   'expo' = exposición teórica según señales (cota superior: los stops dejan más efectivo)")
print("="*70)
display(df_sw.sort_values('Sharpe', ascending=False).style.format(
    {'CAGR': '{:.2%}', 'Vol': '{:.2%}', 'Sharpe': '{:.2f}', 'MaxDD': '{:.1%}', 'expo': '{:.0%}'}))
for nombre, r in ref.items():
    print(f"   Referencia {nombre:<20} CAGR {r['CAGR']:.2%} | Sharpe {r['Sharpe']:.2f} | MaxDD {r['MaxDD']:.1%}")

sh_spy = ref['SPY']['Sharpe']
print(f"\n   Sharpe de las {total} combinaciones: de {df_sw['Sharpe'].min():.2f} a {df_sw['Sharpe'].max():.2f} "
      f"(mediana {df_sw['Sharpe'].median():.2f}); superan a SPY ({sh_spy:.2f}): "
      f"{(df_sw['Sharpe'] > sh_spy).sum()}/{total}")
print(f"   CAGR de las combinaciones: de {df_sw['CAGR'].min():.2%} a {df_sw['CAGR'].max():.2%}; "
      f"superan a SPY ({ref['SPY']['CAGR']:.2%}): {(df_sw['CAGR'] > ref['SPY']['CAGR']).sum()}/{total}")

piv = df_sw.pivot_table(index='umbral', columns='ventana', values='Sharpe', aggfunc='mean')
fig, ax = plt.subplots(figsize=(6, 4))
ax.grid(False)
im = ax.imshow(piv.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(piv.columns)), [f'{c} ses.' for c in piv.columns])
ax.set_yticks(range(len(piv.index)), [f'>{i}%' for i in piv.index])
for (i, j), val in np.ndenumerate(piv.values):
    ax.text(j, i, f'{val:.2f}', ha='center', va='center', color='w')
ax.set_title('Sharpe medio (sobre max. posiciones)\numbral de sorpresa × ventana')
fig.colorbar(im, ax=ax)
fig.tight_layout()
ruta_sw = Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / "09_earnings_surprise_barrido.png"
fig.savefig(ruta_sw, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## VARIANTES: universo × momentum × tamaño de posición

In [ ]:
# %%
# ==============================================================================
# VARIANTES: universo × momentum × tamaño de posición (¿qué aporta cada cambio?)
# ==============================================================================
# 12 combinaciones con el mismo motor, costes y parámetros. La fila  large / no / fijo  es la
# estrategia de la v5. Es un análisis EXPLORATORIO: con 12 variantes, la mejor lo será en
# parte por azar. Fíjate en la dirección del efecto de cada factor, no en el máximo.
from itertools import product

filas = []
for uni, mom, dina in product(['large', 'mid', 'ambos'], [False, True], [False, True]):
    sen = generar_senales(momentum=mom, universo=uni)
    serie, motor = _backtest_rapido(sen, MAX_POSICIONES, tamano=dina, devolver_motor=True)
    r = _resumen_serie(serie)
    det = detalle_eventos(sen, precios_sig, bench_serie.reindex(precios_sig.index).ffill(), (60,))
    ex = det['extra_60'].dropna() if len(det) else pd.Series(dtype=float)
    t = ex.mean() / (ex.std(ddof=1) / np.sqrt(len(ex))) if len(ex) > 2 else np.nan
    h = motor.estado_final.get('hist_exposicion', {})
    filas.append(dict(universo=uni, momentum='sí' if mom else 'no', tamaño='dinámico' if dina else 'fijo',
                      episodios=len(det), extra_60=ex.mean() if len(ex) else np.nan, t_extra60=t,
                      CAGR=r['CAGR'], Sharpe=r['Sharpe'], MaxDD=r['MaxDD'],
                      expo=float(np.mean(list(h.values()))) if h else np.nan))
df_var = pd.DataFrame(filas)

print("="*70)
print("📊 VARIANTES (exposición 'expo' = real media; extra_60 = exceso sobre lo normal de la acción a 60 ses.)")
print("="*70)
display(df_var.style.format({'extra_60': '{:.2%}', 't_extra60': '{:.2f}', 'CAGR': '{:.2%}', 'Sharpe': '{:.2f}',
                             'MaxDD': '{:.1%}', 'expo': '{:.0%}'}))
r_spy = _resumen_serie(bench_serie.dropna())
print(f"   Referencia SPY: CAGR {r_spy['CAGR']:.2%} | Sharpe {r_spy['Sharpe']:.2f} | MaxDD {r_spy['MaxDD']:.1%}")

def _efecto(factor, a, b):
    """Diferencia media (b - a) emparejando el resto de factores."""
    otros = [c for c in ('universo', 'momentum', 'tamaño') if c != factor]
    p = df_var.pivot_table(index=otros, columns=factor, values=['Sharpe', 'extra_60', 'CAGR'])
    return {m: float((p[m][b] - p[m][a]).mean()) for m in ('Sharpe', 'extra_60', 'CAGR')}

print("\n   Efecto medio de cada cambio (emparejado con el resto de factores):")
for nombre, factor, a, b in (("Filtro de momentum (no → sí)", 'momentum', 'no', 'sí'),
                             ("Tamaño dinámico (fijo → dinámico)", 'tamaño', 'fijo', 'dinámico'),
                             ("Universo (large → ambos)", 'universo', 'large', 'ambos'),
                             ("Universo (large → mid)", 'universo', 'large', 'mid')):
    e = _efecto(factor, a, b)
    print(f"   • {nombre}: ΔSharpe {e['Sharpe']:+.2f} | ΔCAGR {e['CAGR']:+.2%} | Δextra_60 {e['extra_60']:+.2%}")
print("   (extra_60 mide la calidad de la señal; Sharpe/CAGR mezclan señal con exposición y tamaño.)")

## COSTES REALISTAS: ¿cuánto aguanta la estrategia?

In [ ]:
# %%
# ==============================================================================
# COSTES REALISTAS: ¿cuánto aguanta la estrategia?
# ==============================================================================
# El motor cobra  coste × rotación unilateral  (Σ|Δpeso|/2 × patrimonio). Si se opera
# Σ|Δpeso| × patrimonio, el coste por LADO es la mitad del parámetro: 0.10% ≈ 5 pb/lado.
# Aquí se barren varios niveles, incluido uno "mezcla" ponderado por cuánta rotación
# corresponde a grandes y a mid-caps. No se modela divisa (EUR/USD) ni impuestos.

def reconstruir_pesos_objetivo(señales, max_pos, tamano):
    """Repite las decisiones del generador (mismas fechas que el motor) para ver los pesos objetivo."""
    gen = crear_generador_pesos(señales, BENCHMARK, max_pos, STOP_LOSS_PCT,
                                tamano_dinamico=tamano, tramos=TRAMOS_PESO, expo_max=EXPOSICION_MAXIMA)
    motor = MotorBacktestDinamico(
        precios=gestor.precios_ajustados, rentabilidades=gestor.rentabilidades_diarias,
        generador_pesos=gen, frecuencia_decision=FREQ_REBALANCEO, coste_transaccion=0.0,
        benchmark_ticker=None, permitir_short=False, max_short_porcentaje=0.0)
    fechas = gestor.rentabilidades_diarias.index
    estado, prev = {}, fechas[0]
    w, estado = gen(prev, gestor.precios_ajustados.loc[:prev], None, estado)
    filas = {prev: w}
    for f in fechas[1:]:
        if motor._es_fecha_decision(f, prev):
            w, estado = gen(f, gestor.precios_ajustados.loc[:f], None, estado)
            filas[f] = w
            prev = f
    return pd.DataFrame(filas).T.fillna(0.0)

W = reconstruir_pesos_objetivo(señales_finales, MAX_POSICIONES, TAMANO_DINAMICO)
dW = W.diff().abs()
dW.iloc[0] = W.iloc[0].abs()
rot_col = dW.sum()
grupo_w = pd.Series({c: grupo_col.get(c) for c in W.columns})
share_mid = float(rot_col[grupo_w == 'mid'].sum() / rot_col.sum()) if rot_col.sum() > 0 else 0.0
c_mezcla = (1 - share_mid) * COSTE_LARGE + share_mid * COSTE_MID
print(f"Reparto de la rotación objetivo: {1 - share_mid:.0%} grandes / {share_mid:.0%} mid-caps "
      f"→ coste 'mezcla' = {c_mezcla:.3%} (≈ {c_mezcla * 5000:.1f} pb/lado)")

etiquetas = {0.0: 'Sin costes', COSTE_TRANSACCION: 'Base del notebook', COSTE_LARGE: 'Realista: grandes',
             COSTE_MID: 'Realista: mid-caps', c_mezcla: 'Realista: mezcla de la cartera', 0.012: 'Estrés'}
tasas = sorted(set([0.0, 0.0005, 0.001, 0.0015, 0.0025, 0.004, 0.006, 0.008, 0.012, COSTE_TRANSACCION,
                    COSTE_LARGE, COSTE_MID, round(c_mezcla, 6)]))
etiquetas = {round(k, 6): v for k, v in etiquetas.items()}

filas = []
for c in tasas:
    serie, motor = _backtest_rapido(señales_finales, MAX_POSICIONES, coste=c, devolver_motor=True)
    r = _resumen_serie(serie)
    log = pd.DataFrame(motor.log_transacciones)
    anios = (serie.index[-1] - serie.index[0]).days / 365.25
    rot = (log['Rotacion %'].str.rstrip('%').astype(float) / 100).sum() if len(log) else 0.0
    cost_tot = log['Coste (€)'].sum() if len(log) else 0.0
    filas.append(dict(escenario=etiquetas.get(round(c, 6), ''), coste_motor=c, pb_por_lado=c * 5000,
                      CAGR=r['CAGR'], Sharpe=r['Sharpe'], MaxDD=r['MaxDD'], patrimonio_final=serie.iloc[-1],
                      rotacion_anual=rot / anios, coste_anual_pct=cost_tot / serie.mean() / anios))
df_cost = pd.DataFrame(filas)

print("\n" + "="*70)
print("📊 SENSIBILIDAD A LOS COSTES (estrategia principal)")
print("="*70)
display(df_cost.style.format({'coste_motor': '{:.3%}', 'pb_por_lado': '{:.1f}', 'CAGR': '{:.2%}', 'Sharpe': '{:.2f}',
                              'MaxDD': '{:.1%}', 'patrimonio_final': '{:,.0f} €', 'rotacion_anual': '{:.0%}',
                              'coste_anual_pct': '{:.2%}'}))
r_spy = _resumen_serie(bench_serie.dropna())
print(f"   Referencia SPY: CAGR {r_spy['CAGR']:.2%} | Sharpe {r_spy['Sharpe']:.2f}")
print("   rotacion_anual = rotación unilateral por año (1.0 = renovar toda la cartera una vez al año).")
print("   coste_anual_pct = costes pagados por año como % del patrimonio medio.")

def _cruce(x, y, ref):
    """Primer coste al que y cae por debajo de ref (interpolación lineal)."""
    for k in range(len(y)):
        if y[k] < ref:
            if k == 0:
                return 0.0
            return x[k - 1] + (ref - y[k - 1]) * (x[k] - x[k - 1]) / (y[k] - y[k - 1])
    return None

for met, ref, nombre in (('CAGR', r_spy['CAGR'], 'CAGR'), ('Sharpe', r_spy['Sharpe'], 'Sharpe')):
    c_eq = _cruce(df_cost['coste_motor'].to_numpy(), df_cost[met].to_numpy(), ref)
    if c_eq is None:
        print(f"   • El {nombre} no cae por debajo del de SPY ni con {df_cost['coste_motor'].max():.2%}.")
    elif c_eq == 0.0:
        print(f"   • El {nombre} ya es inferior al de SPY sin costes.")
    else:
        print(f"   • Coste de equilibrio ({nombre} = SPY): {c_eq:.2%} en unidades del motor (≈ {c_eq * 5000:.0f} pb/lado).")

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for ax, met in zip(axs, ('CAGR', 'Sharpe')):
    ax.plot(df_cost['coste_motor'] * 100, df_cost[met], marker='o', color='tab:purple', label='Estrategia')
    ax.axhline(r_spy[met], color='tab:red', ls='-.', label='SPY')
    ax.axvspan(COSTE_LARGE * 100, COSTE_MID * 100, color='tab:orange', alpha=.15, label='Rango realista')
    ax.set_xlabel('Coste del motor (% por unidad de rotación)')
    ax.set_title(met)
    ax.legend()
fig.tight_layout()
ruta_c = Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / "09_earnings_surprise_costes.png"
ruta_c.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ruta_c, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

## ANÁLISIS DE SUBGRUPOS: ¿dónde funciona mejor la señal?

In [ ]:
# %%
# ==============================================================================
# ANÁLISIS DE SUBGRUPOS: ¿dónde funciona mejor la señal?
# ==============================================================================
# Se usa el conjunto de eventos SIN el filtro de momentum, para poder comparar también
# "tendencia previa alcista / bajista". La métrica es el exceso EXTRA: rentabilidad sobre SPY
# tras la entrada menos lo que esa misma acción rinde sobre SPY en cualquier fecha. Así se
# aísla la señal del sesgo de supervivencia. Con tantos cortes, algunos saldrán "significativos"
# por azar (se estima abajo): confirma cualquier hallazgo en otros datos antes de creerlo.
HORIZ_SUB = (20, 60)
bench_al = bench_serie.reindex(precios_sig.index).ffill()

sen_sub = generar_senales(momentum=False)
det = detalle_eventos(sen_sub, precios_sig, bench_al, HORIZ_SUB)

if det.empty:
    print("ℹ️ Sin episodios para analizar.")
else:
    ma200_b = bench_al.rolling(200).mean()
    reg = np.where(ma200_b.isna(), 'sin dato', np.where(bench_al > ma200_b, 'alcista', 'bajista'))
    col_pos = {c: k for k, c in enumerate(precios_sig.columns)}
    tend_arr = tendencia_sig.to_numpy()
    etq_q = {1: 'Q4 (ene–mar)', 2: 'Q4 (ene–mar)', 3: 'Q4 (ene–mar)', 4: 'Q1 (abr–jun)', 5: 'Q1 (abr–jun)',
             6: 'Q1 (abr–jun)', 7: 'Q2 (jul–sep)', 8: 'Q2 (jul–sep)', 9: 'Q2 (jul–sep)',
             10: 'Q3 (oct–dic)', 11: 'Q3 (oct–dic)', 12: 'Q3 (oct–dic)'}
    det['sector'] = det['activo'].map(sector_col)
    det['tamaño'] = det['activo'].map(grupo_col)
    det['régimen de mercado'] = [reg[max(i - 2, 0)] for i in det['i']]            # SPY vs MA200 en T-1
    det['trimestre del anuncio'] = [etq_q[precios_sig.index[max(i - 1, 0)].month] for i in det['i']]
    det['tramo de sorpresa'] = pd.cut(det['sorpresa'], [-np.inf, 5, 10, np.inf], labels=['< 5%', '5–10%', '> 10%'])
    tv = [tend_arr[max(i - 2, 0), col_pos[a]] for i, a in zip(det['i'], det['activo'])]
    det['tendencia previa'] = np.where(np.isnan(tv), 'sin dato', np.where(np.array(tv) > 0, 'alcista', 'bajista'))

    def _t(s):
        return s.mean() / (s.std(ddof=1) / np.sqrt(len(s))) if len(s) > 2 and s.std(ddof=1) > 0 else np.nan

    def resumen_sub(dim, h):
        g = det.dropna(subset=[f'extra_{h}']).groupby(dim, observed=True)[f'extra_{h}']
        return pd.DataFrame({'n': g.size(), 'medio': g.mean(), 'acierto': g.apply(lambda s: (s > 0).mean()), 't': g.apply(_t)})

    dims = ['sector', 'tamaño', 'régimen de mercado', 'trimestre del anuncio', 'tramo de sorpresa', 'tendencia previa']
    print("="*70)
    print(f"📊 SUBGRUPOS — {len(det)} episodios (sin filtro de momentum), universo '{UNIVERSO_ACTIVO}'")
    print("="*70)
    n_celdas, n_sig = 0, 0
    tablas = {}
    for dim in dims:
        t20, t60 = resumen_sub(dim, 20), resumen_sub(dim, 60)
        tab = t60.join(t20, lsuffix='_60', rsuffix='_20')[['n_60', 'medio_20', 't_20', 'medio_60', 'acierto_60', 't_60']]
        tab.columns = ['n', 'extra 20 ses.', 't (20)', 'extra 60 ses.', 'acierto 60', 't (60)']
        tablas[dim] = tab
        n_celdas += int(tab['t (60)'].notna().sum())
        n_sig += int((tab['t (60)'].abs() > 1.96).sum())
        print(f"\n▶ {dim}")
        display(tab.style.format({'extra 20 ses.': '{:.2%}', 't (20)': '{:.2f}', 'extra 60 ses.': '{:.2%}',
                                  'acierto 60': '{:.0%}', 't (60)': '{:.2f}'}))
    print(f"\n   Subgrupos evaluados a 60 ses.: {n_celdas}; con |t| > 1.96: {n_sig}. "
          f"Por puro azar cabría esperar ≈ {0.05 * n_celdas:.1f}.")
    print("   Subgrupos con n < 15 no son concluyentes. Los episodios de un mismo trimestre no son independientes.")

    fig, axs = plt.subplots(2, 3, figsize=(15, 8))
    for ax, dim in zip(axs.ravel(), dims):
        g = det.dropna(subset=['extra_60']).groupby(dim, observed=True)['extra_60']
        m, se, n = g.mean() * 100, g.std(ddof=1) / np.sqrt(g.size()) * 100, g.size()
        ax.bar(range(len(m)), m.values, yerr=1.96 * se.fillna(0).values, color='tab:purple', alpha=.75, capsize=3)
        ax.axhline(0, color='k', lw=.8)
        ax.set_xticks(range(len(m)), [f'{k}\n(n={n[k]})' for k in m.index], rotation=30, ha='right', fontsize=8)
        ax.set_title(dim)
        ax.set_ylabel('exceso extra a 60 ses. (%)')
    fig.suptitle('Exceso extra por subgrupo (barras = media; error = IC 95%)')
    fig.tight_layout()
    ruta_s = Path(gestor_proyecto.DIRS["OUTPUTS_FIGURES"]) / "09_earnings_surprise_subgrupos.png"
    ruta_s.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(ruta_s, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# --- Nivel cartera: ¿la estrategia rinde distinto con mercado alcista y bajista? ---
pat = motor_dinamico.serie_patrimonio
r_est = pat.pct_change().dropna()
r_spy_d = bench_al.reindex(pat.index).pct_change().reindex(r_est.index)
ma200_b = bench_al.rolling(200).mean()
alc = (bench_al.shift(1) > ma200_b.shift(1)).reindex(r_est.index)      # régimen conocido al cierre anterior
val = ma200_b.shift(1).reindex(r_est.index).notna()
filas = []
for nombre, m in (('Mercado alcista (SPY > MA200)', alc & val), ('Mercado bajista (SPY < MA200)', (~alc) & val)):
    if m.sum() > 20:
        filas.append(dict(régimen=nombre, días=int(m.sum()), est_anual=r_est[m].mean() * 252,
                          spy_anual=r_spy_d[m].mean() * 252, vol_est=r_est[m].std() * np.sqrt(252),
                          vol_spy=r_spy_d[m].std() * np.sqrt(252)))
if filas:
    df_reg = pd.DataFrame(filas).set_index('régimen')
    df_reg['captura'] = df_reg['est_anual'] / df_reg['spy_anual']
    print("\n" + "="*70)
    print("📊 CARTERA POR RÉGIMEN DE MERCADO (rentabilidades medias anualizadas, sin descontar 'rf')")
    print("="*70)
    display(df_reg.style.format({'est_anual': '{:.1%}', 'spy_anual': '{:.1%}', 'vol_est': '{:.1%}',
                                 'vol_spy': '{:.1%}', 'captura': '{:.2f}'}))
    print("   'Bajista' significa solo SPY bajo su MA200: incluye rebotes violentos (p. ej. 2020), por eso SPY puede")
    print("   mostrar una media anualizada alta en ese régimen. La media aritmética diaria × 252 también sobrestima.")
    print("   captura = rentabilidad media de la estrategia / la de SPY en ese régimen (con SPY negativo, "
          "un valor < 1 significa que la estrategia pierde MÁS que SPY, no menos).")

## BLOQUE 9: CONCLUSIONES PEDAGÓGICAS Y CIERRE

In [ ]:
# ==============================================================================
# BLOQUE 9: CONCLUSIONES PEDAGÓGICAS Y CIERRE
# ==============================================================================
print("\n" + "="*70)
print("🎓 CONCLUSIONES DEL CAPÍTULO 9: EARNINGS SURPRISE")
print("="*70)
_txt_tam = ('peso dinámico ' + '/'.join(f"{p:.0%}" for _, p in TRAMOS_PESO[::-1]) + ' según la sorpresa') if TAMANO_DINAMICO else f"{1/MAX_POSICIONES:.1%} c/u"
print(f"""
📌 LECCIONES CLAVE DE ESTE CAPÍTULO
====================================

1. EL MERCADO INFRARREACCIONA DE FORMA SISTEMÁTICA
   • El PEAD es una de las anomalías más documentadas: los inversores tardan en
     incorporar del todo una sorpresa fundamental, creando un drift aprovechable.
     Su magnitud ha caído con el tiempo en grandes compañías muy seguidas.

2. LA CALIDAD DE LA SEÑAL ES CRÍTICA
   • Aquí exigimos sorpresa > {UMBRAL_SORPRESA}%, reacción del precio positiva frente a
     {BENCHMARK} y volumen anómalo en el anuncio. Un umbral más estricto reduce el
     ruido pero también el número de operaciones: compruébalo en el embudo del Bloque 5.

3. TIMING Y LOOK-AHEAD
   • Entrar el mismo día del anuncio puede usar información que aún no existía
     (anuncios tras el cierre). Por eso la señal empieza en T+1, y se sale
     {MARGEN_PRE_EARNINGS} sesiones antes del siguiente anuncio para no asumir riesgo binario.

4. LIMITACIONES DE LOS DATOS RETAIL
   • `yfinance` es un buen proxy didáctico, pero puede tener huecos y no garantiza
     la hora del anuncio. Además, el universo (compañías actuales) introduce
     sesgo de supervivencia: los resultados tienden a ser optimistas.
     Un sistema profesional usaría Compustat, I/B/E/S o Refinitiv.

5. GESTIÓN DEL RIESGO EVENT-DRIVEN
   • Máx. {MAX_POSICIONES} posiciones ({_txt_tam}), stop-loss del {abs(STOP_LOSS_PCT):.0%} y efectivo cuando
     no hay señales. Pueden pasar semanas sin operar: la disciplina de no forzar
     operaciones fuera de la ventana es parte de la estrategia.

🚪 PRÓXIMO PASO
================
En el Capítulo 10 exploraremos el "Insider Trading Legal", analizando
cómo los movimientos de los directivos (Form 4 en la SEC) pueden actuar
como una señal de confirmación de alto valor para nuestras estrategias.

🎯 RECUERDA
============
El Earnings Surprise no es una estrategia de "comprar y olvidar".
Es un satélite táctico. Úsalo para potenciar convicciones ya existentes
o como un sistema sistemático de rotación que captura el momentum
fundamental post-anuncio.
""")
print("="*70)
print("✅ CAPÍTULO 9 COMPLETADO")
print("="*70)